# Run and inspect the IDS study in Google Colab

1. Download `cybersecurity_ids_extension.zip` from the project files and upload this notebook to Colab.
2. Run the cells from top to bottom. The first cell asks you to upload the ZIP.
3. The notebook installs the dependencies, reruns the experiment, and displays holdout/external-test metrics and plots. CPU runtime is sufficient; expect a few minutes.

The synthetic mode is not used here: the ZIP contains the NSL-KDD data files used for the reported run.

In [ ]:
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()  # Choose cybersecurity_ids_extension.zip
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
PROJECT = pathlib.Path('/content/cybersecurity_ids_extension')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(PROJECT)
assert (PROJECT / 'cybersecurity_ids_study.py').exists()
assert (PROJECT / 'data' / 'KDDTrain+_20Percent.txt').exists()
print('Project ready at:', PROJECT)
print('Training file rows:', sum(1 for _ in open(PROJECT / 'data' / 'KDDTrain+_20Percent.txt')))

In [ ]:
# Install the tested core dependencies and optional XGBoost/LightGBM learners.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT / 'requirements.txt'),
    '-r', str(PROJECT / 'requirements-optional.txt'),
], check=True)
print('Dependencies installed. If Colab requests a runtime restart, restart and rerun from the upload cell.')

In [ ]:
# Train, tune, evaluate, and regenerate the CSVs/figures.
import subprocess, sys
command = [
    sys.executable, 'cybersecurity_ids_study.py',
    '--out', 'results', '--cv', '3', '--n-iter', '2', '--target-recall', '0.995',
]
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
# Inspect fixed-default-threshold metrics on the paper-like holdout.
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(PROJECT / 'results' / 'metrics.csv')
metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
holdout = metrics[(metrics['dataset'] == 'paper-like holdout') & (metrics['threshold_mode'] == 'default')].copy()
holdout[metric_cols] = (100 * holdout[metric_cols]).round(3)
display(holdout[['model', 'threshold', *metric_cols, 'tn', 'fp', 'fn', 'tp']].sort_values('f1', ascending=False))
print('Percent columns are percentages; FN is the missed-attack count.')

In [ ]:
# Inspect the untouched KDDTest+ external evaluation.
external = metrics[(metrics['dataset'] == 'official KDDTest+ external') & (metrics['threshold_mode'] == 'default')].copy()
external[metric_cols] = (100 * external[metric_cols]).round(3)
display(external[['model', 'threshold', *metric_cols, 'tn', 'fp', 'fn', 'tp']].sort_values('f1', ascending=False))

# Recall-oriented operating points (selected using training-only calibration data).
operational = metrics[(metrics['dataset'].isin(['paper-like holdout', 'official KDDTest+ external'])) & (metrics['threshold_mode'] == 'operational')].copy()
operational[metric_cols] = (100 * operational[metric_cols]).round(3)
display(operational[['dataset', 'model', 'threshold', *metric_cols, 'tn', 'fp', 'fn', 'tp']].sort_values(['dataset', 'recall'], ascending=[True, False]))

In [ ]:
# Cross-validation mean ± standard deviation, in percent.
cv = pd.read_csv(PROJECT / 'results' / 'cross_validation.csv')
for column in cv.columns:
    if column.startswith('cv_') and column.endswith(('_mean', '_std')):
        cv[column] = (100 * cv[column]).round(3)
display(cv.sort_values('cv_f2_mean', ascending=False))

In [ ]:
# View the generated ROC, permutation-importance, and confusion-matrix plots.
from IPython.display import Image, display
for filename in ['roc_curves.png', 'permutation_importance.png', 'confusion_matrices_holdout.png']:
    print(filename)
    display(Image(filename=str(PROJECT / 'results' / 'figures' / filename)))